## Building the focus dataset


In this notebook we use the network learned on images cropped in the bounding boxes to generate a new dataset. From the original data set `Imagenet_full` we will fixate on the most likely position and generate the `Imagenet_focus` dataset

TODO: use rotations of the image to find the center of the object


In [1]:
import retinoto_py as fovea
args = fovea.Params(do_fovea=True, batch_size=1, shuffle=False)
print(args)

Params(batch_size=1, num_workers=0, prefetch_factor=0, image_size=224, grid_size_ecc=161, grid_size_ang=309, do_mask=False, do_fovea=True, use_hexagonal_grid=True, rs_min=-3.5, rs_max=0.7, angle_start=-5.235987755982989, angle_margin=0.010166966516471823, mode='bilinear', padding_mode='zeros', model_name='convnext_base', num_epochs=100, subset_factor=1, optimizer_name='adamw', loss_name='CrossEntropyLoss', base_lr=1e-06, final_lr=1e-09, num_warmup_epochs=20, delta1=0.3, delta2=0.002, weight_decay=0.0001, label_smoothing=0.0005, do_full_training=True, do_augment=True, augment_proba=0.85, stochastic_depth_prob=0.6, seed=1998, shuffle=False, verbose=False)


In [2]:
resolution = (21, 34) # landscape images
# resolution = (13, 21) # landscape images
resolution = (13, 13)
N_fixations = fovea.np.prod(resolution)

size_ratios = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
size_ratios = fovea.np.linspace(0.8, 1.0, 3, endpoint=True)
size_ratios = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

angles = [0, 45, 90, 135, 180]
angles = [0, 60, 120]

format = 'png'
subset_factor = 1


In [3]:
# x and y are coordinates in the image, with (0,0) being the bottom-left corner. x is horizontal, y is vertical
def get_boxes(df, value):
    idx = list(df['ImageId'][df['ImageId'] == value].index)
    bboxes = []
    if idx:
        for i in range(len(df["PredictionString"][idx[0]].split(' '))//5):
            pos =(5*i)
            bboxes.append({'xmin' : int(df["PredictionString"][idx[0]].split(' ')[1 + pos]),
                           'ymin' : int(df["PredictionString"][idx[0]].split(' ')[2 + pos]),
                           'xmax' : int(df["PredictionString"][idx[0]].split(' ')[3 + (5*i)]),
                           'ymax' : int(df["PredictionString"][idx[0]].split(' ')[4 + (5*i)])
                                        })
    return bboxes

In [4]:
def clean_list(list_dir, EXCLUDED_FILES={'.DS_Store', '.ipynb_checkpoints'}):
    return [ p for p in list_dir if p.is_file() and p.name not in EXCLUDED_FILES  ]


In [ ]:
from torchvision.io import read_image
from torchvision.utils import save_image

FULL_DATA_DIR = args.DATAROOT / 'Imagenet_full'
FOCUS_DATA_DIR = args.DATAROOT / 'Imagenet_focus'
FOCUS_DATA_DIR.mkdir(exist_ok=True)
IMG_EXTS = {'.jpg', '.jpeg', '.JPEG', '.png', '.bmp'}

args = fovea.Params(do_fovea=True, model_name='convnext_base', batch_size=1, shuffle=False, subset_factor=subset_factor)
model = fovea.load_model(args, model_filename=args.data_cache / f'32_fovea_model_name=convnext_base_dataset=bbox.pth')

for folder in ['val', 'train']:
    print(f'\n Scanning folder "{folder}"')
    annotation_file = args.DATAROOT / f'LOC_{folder}_solution.csv'
    with open(annotation_file, 'r') as csv_file:
        df_data = fovea.pd.read_csv(csv_file)

    src_root = FULL_DATA_DIR / folder
    tgt_root = FOCUS_DATA_DIR / folder
    tgt_root.mkdir(parents=True, exist_ok=True)

    dataset = fovea.get_dataset(args, src_root, do_full_preprocess=False)
    loader = fovea.get_loader(args, dataset)
    class_to_idx = dataset.class_to_idx

    count_in = 0
    count_out = 0

    # parcours récursif avec pathlib
    for img_path in fovea.tqdm(clean_list(list(src_root.rglob('*.*')))):
        if not img_path.is_file() or img_path.suffix not in IMG_EXTS:
            print(f'File {img_path} is detected as an invalid image.')
            continue

        count_in += 1

        lock_filename = img_path.with_suffix('.lock')

        if not lock_filename.exists():
            lock_filename.touch(exist_ok=True)

            imgid = img_path.stem

            class_id = img_path.parent.name
            true_idx = class_to_idx[class_id]
            target_folder = tgt_root / class_id
            target_folder.mkdir(parents=True, exist_ok=True)
            out_path = target_folder / f'{imgid}.{format}'

            try:
                image = read_image(img_path)/255.
                if image.shape[0] == 1: image = image.repeat(3, 1, 1)
                three, H, W = image.shape
                assert three == 3

            except Exception as e:
                print(f' could not open {img_path}: {e}')
                break
            image = image.squeeze(0)

            # put the positions inside the bounding box of the object, instead of the whole image
            boxes = get_boxes(df_data, imgid)
            if not boxes:
                boxes = [{'xmax': H, 'xmin': 0, 'ymax': W, 'ymin': 0}]

            for i_obj, b in enumerate(boxes):
                try:
                    xmin, ymin, xmax, ymax = b['xmin'], b['ymin'], b['xmax'], b['ymax']

                    no = '' if i_obj == 0 else f'_{i_obj}'
                    out_path = target_folder / f'{imgid}{no}.{format}'
                    if out_path.is_file():
                        # the file already exists let's skip it
                        count_out += 1
                        continue

                    # aspect_ratio = (b['xmax']-b['xmin'])/(b['ymax']-b['ymin'])
                    # resolution = (int(np.sqrt(N_fixations*aspect_ratio)), int(np.sqrt(N_fixations/aspect_ratio)))

                    # pos_H, pos_W = fovea.get_positions(b['xmax']-b['xmin'], b['ymax']-b['ymin'], resolution=resolution)
                    # pos_H, pos_W = pos_H + b['xmin'], pos_W + b['ymin']


                    h_center, w_center = (ymin+ymax)//2, (xmin+xmax)//2
                    box_size = min((H, W))
                    # box_size = min((box_size, int(1.3*max(((xmax-xmin), (ymax-ymin))))))
                    box_size = min((box_size, max(((xmax-xmin), (ymax-ymin)))))
                    pos_H, pos_W = fovea.get_positions(box_size, box_size, resolution=resolution)
                    pos_H, pos_W = pos_H + h_center - box_size/2, pos_W + w_center - box_size/2

                    size_ratio_max = size_ratios[0]
                    likelihood_map_label_maximum = 0
                    likelihood_map_label = None
                    for size_ratio in size_ratios:
                        for angle in angles:
                            probas = fovea.compute_likelihood_map(args, model, image, pos_H, pos_W, 
                                                                size_ratio=size_ratio, angle=angle, do_softmax=True)
                            probas = probas.cpu()
                            likelihood_map_label_ = probas[:, true_idx]

                            if likelihood_map_label_.max() > likelihood_map_label_maximum:
                                likelihood_map_label_maximum = likelihood_map_label_.max()
                                size_ratio_max = size_ratio

                            if likelihood_map_label is None:
                                likelihood_map_label = likelihood_map_label_
                            else:
                                likelihood_map_label += likelihood_map_label_


                    likelihood_max, idx_pos = likelihood_map_label.max(axis=-1)

                    if likelihood_max < 0.25:
                        print(f' - low likelihood for {img_path}: {likelihood_max:.3f}')
                    else:
                        box_size = int(fovea.np.sqrt(H*W)*size_ratio_max)
                        image_fix = fovea.fixate(image, int(pos_H[idx_pos]), int(pos_W[idx_pos]), box_size) 

                        # img_pil = fovea.TF.to_pil_image(image_fix)
                        # img_pil.save(out_path, format=format)
                        save_image(image_fix, out_path)
                except Exception as e:
                    print(e)

            lock_filename.unlink(missing_ok=True)

            count_out += 1

    print(f' - in: {count_in} / out: {count_out}')


 Scanning folder "val"


  0%|          | 0/50006 [00:00<?, ?it/s]

Argument #6: Padding size should be less than the corresponding input dimension, but got: padding (0, 102) at dimension 2 of input [1, 3, 60, 104]


In [ ]:
pos_H, pos_W, H, W, box_size

In [ ]:
boxes

In [ ]:
boxes['xmax']-boxes['xmin']


Voilà !